In [2]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pandas", "numpy", "networkx"])

  Using cached pandas-2.3.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached numpy-2.3.4-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.3.3-cp314-cp314-macosx_11_0_arm64.whl (10.8 MB)
Using cached numpy-2.3.4-cp314-cp314-macosx_14_0_arm64.whl (5.1 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]2m3/4 [pandas]



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


0

In [3]:
import pandas as pd
import numpy as np

df_alters = pd.read_csv('./data/df_alter.csv')

# distance = np.linalg.norm([x - 0.5, y - 0.5])
# apply for rows rather than columns is axis=1
df_alters['distance'] = df_alters.apply(lambda row: np.linalg.norm([row['user_layout_x'] - 0.5, row['user_layout_y'] - 0.5]), axis=1)

# averages of Euclidean distance for every alter
df_concepts_2 = df_alters[["name","distance"]].groupby("name").agg(["count","mean"])
df_concepts_2.columns = ["mentions","distance"]
# Sort the concepts so that least distance is on top.
df_concepts_2.sort_values("distance",ascending=False, inplace=True)
df_concepts_2.head(5)

,mentions,distance
name,,
user experience,2,0.284306
art,1,0.266815
race-ethnicity,2,0.244463
optimisation,6,0.230239
social networks,7,0.227122


In [4]:
# non-averaged Euclidean distance broken down by user
alter_distance_by_user = df_alters.loc[df_alters['name'] == 'race-ethnicity']
alter_distance_by_user[["name", "secret_word", "distance"]]

,name,secret_word,distance
29,race-ethnicity,porcupine,0.188387
113,race-ethnicity,helloOII,0.300540


## Claim: 
The merged network uses the average Euclidean distance of an alter from the ego. 

There are only 2 people who put "race-ethnicity" on their sociograms, and because their dstances are far apart and the sample size is small (meaning that *porcupine* placed "race-ethnicity" closer to the center and *helloOII* placed it further) the averaged distance does a poor job of merging these positions. 

In [ ]:
import networkx as nx

unique_secret_words = df_alters['secret_word'].unique()
print(unique_secret_words)

food_secret_words = ['falafel', 'marmite', 'peaches', 'peanuts', 'apples', 'cookie']

food = {}

for i in food_secret_words:
    alter_distance_by_user = df_alters.loc[df_alters['secret_word'] == i]
    smallest_distance = alter_distance_by_user.nsmallest(3, 'distance')
    name = str(smallest_distance["name"].values[0:3])
    food[i] = name

print(food)

#
# {'falafel': "['empirical data' 'ethics' 'personal']" #methods
# 'marmite': "['culture' 'freedom' 'community']", # social science, data science, social science
# 'peaches': "['identity' 'structure' 'theory']", #ALL guiding concepts



['falafel' 'marmite' 'porcupine' 'cats' 'random' 'space' 'peaches'
 'Dolphin' 'triceratops' 'test' 'helloOII' 'peanuts' 'apples' 'cat' 'Gem'
 'cookie']
{'falafel': "['empirical data' 'ethics' 'personal']", 'marmite': "['culture' 'freedom' 'community']", 'peaches': "['identity' 'structure' 'theory']", 'peanuts': "['AI' 'knowledge' 'analysis']", 'apples': "['context' 'empirical data' 'history']", 'cookie': "['design' 'community' 'perspective']"}
